## UAV Intrusion Detection Stacked Ensemble
This notebook uses One-Class SVM, Autoencoder, and Local Outlier Factor as base models, and combines their outputs with a meta-classifier ensemble.

In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import classification_report
import os

In [16]:
# Constants
RANDOM_STATE = 1082
base_dir = "../../../Datasets/excel/2dSpoofingDataset/"
file_names = [
    "spoofing-CTGAN-synthetic.csv",
    "spoofing-GaussianCopula-synthetic.csv",
    "spoofing-TVAE-synthetic.csv",
]

In [17]:
for file_name in file_names:
    file_path = os.path.join(base_dir, file_name)
    try:
        df = pd.read_csv(file_path)
        print(f"\n--- Processing {file_name} ---")
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        continue
    except Exception as e:
        print(f"An error occurred while loading {file_name}: {e}")
        continue

    if 'Output' not in df.columns:
        print(f"Warning: 'Output' column not found in {file_name}. Skipping.")
        continue

    # Prepare features and labels
    X = df.drop('Output', axis=1)
    y = df['Output']

    # Use benign data to train unsupervised models
    X_train_raw = X[y == 0]
    X_test_raw = X
    y_test = y

    # Normalize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw)

    # PCA for OC-SVM and LOF (85% variance)
    pca_svm_lof = PCA(n_components=0.85, random_state=RANDOM_STATE)
    X_train_pca_svm_lof = pca_svm_lof.fit_transform(X_train_scaled)
    X_test_pca_svm_lof = pca_svm_lof.transform(X_test_scaled)

    # PCA for Autoencoder (95% variance)
    pca_ae = PCA(n_components=0.95, random_state=RANDOM_STATE)
    X_train_pca_ae = pca_ae.fit_transform(X_train_scaled)
    X_test_pca_ae = pca_ae.transform(X_test_scaled)

    # One-Class SVM
    ocsvm = OneClassSVM(nu=0.011, gamma=0.000211)
    ocsvm.fit(X_train_pca_svm_lof)
    ocsvm_score = ocsvm.decision_function(X_test_pca_svm_lof)

    # Local Outlier Factor
    lof = LocalOutlierFactor(n_neighbors=1000, novelty=True)
    lof.fit(X_train_pca_svm_lof)
    lof_score = lof.decision_function(X_test_pca_svm_lof)

    # Autoencoder using MLP
    ae = MLPRegressor(hidden_layer_sizes=(3,), activation='relu', max_iter=100, random_state=RANDOM_STATE)
    ae.fit(X_train_pca_ae, X_train_pca_ae)
    recon = ae.predict(X_test_pca_ae)
    ae_score = np.mean((X_test_pca_ae - recon) ** 2, axis=1)

    # Stack raw anomaly scores
    stacked_features = np.vstack([ocsvm_score, lof_score, ae_score]).T

    # Meta-classifier (Random Forest)
    meta_clf = RandomForestClassifier(random_state=RANDOM_STATE)
    meta_clf.fit(stacked_features, y_test)
    stacked_pred = meta_clf.predict(stacked_features)

    print(classification_report(
        y_test,
        stacked_pred,
        labels=[0, 1, 2, 3],
        target_names=["benign", "attack1", "attack2", "attack3"],
        zero_division=0
    ))


--- Processing spoofing-CTGAN-synthetic.csv ---


C:\Users\antho\miniforge3\envs\sklearn-env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


              precision    recall  f1-score   support

      benign       1.00      1.00      1.00      2962
     attack1       1.00      1.00      1.00      2294
     attack2       1.00      1.00      1.00      2403
     attack3       1.00      1.00      1.00      2341

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000


--- Processing spoofing-GaussianCopula-synthetic.csv ---


C:\Users\antho\miniforge3\envs\sklearn-env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


              precision    recall  f1-score   support

      benign       1.00      1.00      1.00      7791
     attack1       1.00      1.00      1.00       710
     attack2       1.00      1.00      1.00       884
     attack3       1.00      1.00      1.00       615

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000


--- Processing spoofing-TVAE-synthetic.csv ---


C:\Users\antho\miniforge3\envs\sklearn-env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


              precision    recall  f1-score   support

      benign       1.00      1.00      1.00      7844
     attack1       1.00      1.00      1.00      1049
     attack2       1.00      1.00      1.00       680
     attack3       1.00      1.00      1.00       427

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000

